# Directional unlearning: Kaggle launcher

Clone, install, run. No experiment logic lives here; every cell calls a module in the repo.

**Session settings** (right panel, before running):
- Accelerator: **GPU T4 x2** (only one is used) or P100
- Internet: **on** (needed to clone, to pull the Pythia weights, and to fetch the eval corpora)
- Persistence: Variables and Files off; everything is written to `/kaggle/working`

**To run headless**: Save Version -> Save & Run All. The session runs to completion with
no browser attached and `/kaggle/working` is kept as the version's output.

Results land in `/kaggle/working/results`. Download them and commit them from the local
machine, so every results file still traces to the commit it ran from.

### What to watch in Phase 1

Two perplexities print each epoch. **`pile`** is a held-out slice of Pythia's own
pretraining data and is the number the utility gate uses. **`wikitext`** is reported for
continuity, but Phase 1 replay is drawn from its *train* split, so a good score there is
partly domain fit rather than retained ability. Phase 1 fails if `pile` exceeds 1.5x what
the base model scored at the start of the same run.

The open question this run answers: **fact accuracy has to reach 0.90 in both directions
before perplexity crosses the gate.** In a 2-epoch local trial accuracy was climbing
(forward 0.66 -> 0.86) while WikiText perplexity rose 44 -> 57, and it was not clear which
would arrive first over 10 epochs. If perplexity wins, the fix is more replay weight or
fewer epochs, not a looser gate.

In [ ]:
# Pin the commit to run. 'main' is fine for exploration; pin a hash for anything
# whose numbers you intend to keep, so the results file and the code agree.
REPO = "https://github.com/EdanBarrios/directional-unlearning.git"
COMMIT = "main"

import subprocess, sys, os, pathlib

WORK = pathlib.Path("/kaggle/working")
SRC = WORK / "repo"
if not SRC.exists():
    subprocess.run(["git", "clone", "--quiet", REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "fetch", "--quiet", "--all"], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--quiet", COMMIT], check=True)
os.chdir(SRC)
sys.path.insert(0, str(SRC))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Kaggle images ship torch with CUDA already. Only add what is missing, and never
# reinstall torch: that pulls a CPU build and silently removes the GPU.
!pip install --quiet --no-deps 'transformers>=5.0' 'datasets>=4.0' pyyaml tqdm 2>&1 | tail -2

import torch
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
assert torch.cuda.is_available(), "No GPU. Set Accelerator to T4 or P100 in the session settings."

In [ ]:
# Smoke test first, always. Pythia-70m, 5 facts, 3 steps. Under a minute.
# If this fails, nothing below is worth starting.
!SMOKE=1 python -m data.gen_facts
!SMOKE=1 python -m src.probe --out /kaggle/working/results/smoke_probe.json
!SMOKE=1 python -m src.finetune --config configs/phase1.yaml --seed 0 \
    --out /kaggle/working/results/smoke_finetune.json --no-save

In [ ]:
# Time one training step before committing to a long run, so a bad session is caught
# in seconds rather than after an hour. Expect roughly 0.1 s/step on a T4.
import time
from src import data as D
from src.model import load
from src.losses import lm_loss

_m, _t = load("EleutherAI/pythia-160m")
_d = D.load_facts("data/facts.json")
_b = D.encode(_t, D.train_records(_d)[:32], _m.device)
_o = torch.optim.AdamW(_m.parameters(), lr=1e-5)
_m.train()
for _ in range(3):
    _l = lm_loss(_m, _b); _o.zero_grad(set_to_none=True); _l.backward(); _o.step()
torch.cuda.synchronize(); _s = time.time()
for _ in range(10):
    _l = lm_loss(_m, _b); _o.zero_grad(set_to_none=True); _l.backward()
    torch.nn.utils.clip_grad_norm_(_m.parameters(), 1.0); _o.step()
torch.cuda.synchronize()
_step = (time.time() - _s) / 10
print(f"{_step*1000:.0f} ms/step -> {_step*272/60:.1f} min/epoch, {_step*272*10/60:.0f} min per 10-epoch Phase 1 run")
del _m, _o; torch.cuda.empty_cache()

## Phase 1: train M1

Run once. The checkpoint is the starting point for every later phase, so save it as a
Kaggle Dataset (next cell) and never retrain it: all conditions must start from identical
weights or the comparison is between starting points, not treatments.

In [ ]:
!python -m src.finetune --config configs/phase1.yaml --seed 0 --lr 1e-5 \
    --out /kaggle/working/results/phase1/lr1e-05_seed0.json \
    --save /kaggle/working/M1

In [ ]:
# Full probe of M1 with all 149 alternatives: the Phase 1 acceptance table.
!python -m src.probe --model /kaggle/working/M1 \
    --out /kaggle/working/results/phase1/M1_probe.json

In [ ]:
# Did Phase 1 pass? Both criteria must hold: facts learned AND model not destroyed.
import json

r = json.load(open("/kaggle/working/results/phase1/lr1e-05_seed0.json"))
m = r["metrics"]
print(f"success={m['success']}    base ppl={m['base_ppl']}    gate={m['max_ppl']:.2f}")
print(f"final ppl={m['final'].get('ppl_by_corpus')}")
for d in ("d_fwd", "d_rev", "s_fwd"):
    print(f"  {d} " + "  ".join(f"{s}={m['final'][d][s]['accuracy']:.2f}" for s in ("forget", "dose", "retain")))
print("\nper-epoch trade-off (the race between accuracy and perplexity):")
for h in r["history"]:
    e = h["eval"]
    print(f"  epoch {h['epoch']:2d}  ppl={e.get('ppl', float('nan')):7.2f}  "
          f"d_fwd={e['d_fwd']['forget']['accuracy']:.2f}  "
          f"d_rev={e['d_rev']['forget']['accuracy']:.2f}  "
          f"s_fwd={e['s_fwd']['forget']['accuracy']:.2f}")

## Phase 2: unlearn

Run only if Phase 1 reported `success=True`. All six runs start from the same M1, so they
are comparable; that is why M1 is trained once and reused rather than retrained per run.

A run that hits the step cap is marked `unlearned: false` and saves no checkpoint. That is
control C1, and it must be excluded from analysis rather than tuned around.

In [ ]:
for method in ("npo_gd", "ga_gd"):
    for cond in ("u_fwd", "u_both", "u_fwd_dose"):
        print(f"\n===== {method} / {cond} =====")
        !python -m src.unlearn --config configs/{method}.yaml --condition {cond} --seed 0 \
            --checkpoint /kaggle/working/M1 \
            --out /kaggle/working/results/phase2/{method}_{cond}_seed0.json \
            --save /kaggle/working/ckpt/{method}_{cond}_seed0

In [ ]:
# Bundle results for download. Checkpoints are too large to commit; publish M1 as a
# Kaggle Dataset (Data panel -> New Dataset -> from this notebook's output) and attach
# it to later sessions instead of retraining.
!cd /kaggle/working && tar czf results.tar.gz results && du -sh results.tar.gz M1 2>/dev/null
import glob
for f in sorted(glob.glob("/kaggle/working/results/**/*.json", recursive=True)):
    d = json.load(open(f))
    print(f"{f.split('results/')[-1]:44} commit={d.get('git_commit','?')}")